# Project: Predicting diabetes progression

*ML Academy — starter notebook*

**Goal.** Predict disease progression from the baseline measurements, and find out how much better than a naive guess you can do.

**The data.** 442 diabetes patients from the study by Efron, Hastie, Johnstone and Tibshirani (2004), shipped with scikit-learn as load_diabetes. Ten measurements per patient — age, sex, body-mass index (BMI), blood pressure and six blood-serum values (s1–s6) — and a number measuring how far the disease progressed one year later.

**Uses:** Lesson 5, Lesson 6, Lesson 9, Lesson 10.

Work through the steps in order. Lines marked `# TODO` are yours to write; each step ends with a check that prints ✅ when you're right (and a hint when you're not). Cells that use a step's result will show errors until that step is done — that's expected. The solution notebook is in the same folder.

Requirements: `pip install numpy pandas scikit-learn matplotlib scipy` — the dataset ships with scikit-learn, so no download is needed.

In [1]:
# Run this cell first: it defines check(), which tells you whether each step is right.
import numpy as np
def check(get, want, hint=''):
    try:
        got = get()
        if isinstance(want, str):
            ok = str(got) == want
        else:
            ok = np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=1e-6)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


## Step 1. Load and look at the data

Load the data as a pandas table. `scaled=False` gives the original units (age in years, BMI in kg/m², …).

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
print(df.shape)
df.head()

(442, 11)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0,151.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0,75.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0,141.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0,206.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0,135.0


In [3]:
check(lambda: df.shape, [442, 11], hint='Use load_diabetes(as_frame=True, scaled=False).frame.')

✅ Correct!


## Step 2. Split the data and set a baseline

Hold out 20% for testing (`random_state=42`). Then compute the **baseline**: predict the *training* mean for every test patient and measure the MAE. Every real model must beat this number.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
X, y = df.drop(columns="target"), df["target"]
X_train, X_test, y_train, y_test = None, None, None, None  # TODO: split with test_size=0.2 and random_state=42
baseline_pred = None  # TODO: an array holding the training mean, one value per test row
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print("baseline MAE:", round(baseline_mae, 1))

InvalidParameterError: The 'y_true' parameter of mean_absolute_error must be an array-like. Got None instead.

In [5]:
check(lambda: X_train.shape, [353, 10], hint='train_test_split(X, y, test_size=0.2, random_state=42).')
check(lambda: baseline_mae, 64.00646146990485, hint='np.full(len(y_test), y_train.mean()) repeats the training mean.')

❌ Not yet. Hint: train_test_split(X, y, test_size=0.2, random_state=42).
❌ Not yet. Hint: np.full(len(y_test), y_train.mean()) repeats the training mean.


## Step 3. One feature: BMI

Fit a straight line using BMI alone (Lesson 6). How much of the baseline error does it remove?

In [6]:
from sklearn.linear_model import LinearRegression
bmi_model = None  # TODO: fit LinearRegression on the bmi column only
bmi_pred = bmi_model.predict(X_test[["bmi"]])
bmi_mae, bmi_r2 = mean_absolute_error(y_test, bmi_pred), r2_score(y_test, bmi_pred)
print(f"slope {bmi_model.coef_[0]:.2f} per BMI unit · MAE {bmi_mae:.1f} · R² {bmi_r2:.3f}")

AttributeError: 'NoneType' object has no attribute 'predict'

In [7]:
check(lambda: bmi_mae, 52.259976445345536, hint="Use X_train[['bmi']] (double brackets keep it a table).")
check(lambda: bmi_r2, 0.23335039815872138, hint='r2_score(y_test, bmi_pred).')

❌ Not yet. Hint: Use X_train[['bmi']] (double brackets keep it a table).
❌ Not yet. Hint: r2_score(y_test, bmi_pred).


## Step 4. All ten features

Now use every feature. Put a `StandardScaler` in a pipeline so the coefficients are comparable (Lesson 20).

In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
full_model = None  # TODO: a pipeline of StandardScaler then LinearRegression
full_model.fit(X_train, y_train)
full_pred = full_model.predict(X_test)
full_mae, full_r2 = mean_absolute_error(y_test, full_pred), r2_score(y_test, full_pred)
print(f"MAE {full_mae:.1f} · R² {full_r2:.3f}")
pd.Series(full_model[-1].coef_, X.columns).sort_values().round(1)

AttributeError: 'NoneType' object has no attribute 'fit'

In [9]:
check(lambda: full_mae, 42.79409467959994, hint='make_pipeline(StandardScaler(), LinearRegression()).')
check(lambda: full_r2, 0.4526027629719195, hint='Fit the pipeline on X_train, y_train first.')

❌ Not yet. Hint: make_pipeline(StandardScaler(), LinearRegression()).
❌ Not yet. Hint: Fit the pipeline on X_train, y_train first.


## Step 5. Don't over-read the coefficients

Look at s1 and s2: one has a large negative weight, the other a large positive one. Are they correlated? When two features move together, linear regression can split credit between them in strange ways — the individual weights become unreliable even if predictions are fine (Lesson 10).

In [10]:
corr_s1_s2 = None  # TODO: the correlation between s1 and s2 in the training data
print("correlation of s1 and s2:", round(corr_s1_s2, 3))

TypeError: type NoneType doesn't define __round__ method

In [11]:
check(lambda: corr_s1_s2, 0.8910632483212737, hint="pandas: X_train['s1'].corr(X_train['s2']).")

❌ Not yet. Hint: pandas: X_train['s1'].corr(X_train['s2']).


## Step 6. Ridge, with the penalty chosen by cross-validation

`RidgeCV` tries several penalty strengths with cross-validation and keeps the best (Lesson 10).

In [12]:
from sklearn.linear_model import RidgeCV
ridge = None  # TODO: a pipeline of StandardScaler then RidgeCV(alphas=np.logspace(-3, 3, 13))
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_mae = mean_absolute_error(y_test, ridge_pred)
print("chosen alpha:", ridge[-1].alpha_, "· test MAE:", round(ridge_mae, 1))

AttributeError: 'NoneType' object has no attribute 'fit'

In [13]:
check(lambda: ridge_mae, 42.81199941834887, hint='Same as the full model, but RidgeCV(alphas=np.logspace(-3, 3, 13)) as the last step.')

❌ Not yet. Hint: Same as the full model, but RidgeCV(alphas=np.logspace(-3, 3, 13)) as the last step.


## Step 7. Compare fairly with cross-validation

One test split is noisy. Compare the two models with 5-fold cross-validation on the training set (Lesson 5).

In [14]:
from sklearn.model_selection import cross_val_score
cv_lin = None  # TODO: 5-fold CV MAE for full_model (note the minus sign)
cv_ridge = -cross_val_score(ridge, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
print(f"linear: {cv_lin.mean():.1f} ± {cv_lin.std():.1f}   ridge: {cv_ridge.mean():.1f} ± {cv_ridge.std():.1f}")

InvalidParameterError: The 'estimator' parameter of cross_val_score must be an object implementing 'fit'. Got None instead.

In [15]:
check(lambda: cv_lin.mean(), 45.54332900504238, hint="cross_val_score(full_model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error'), negated.")

❌ Not yet. Hint: cross_val_score(full_model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error'), negated.


## Go further

- Add squared terms with PolynomialFeatures(degree=2) in the pipeline. Does cross-validated MAE improve?
- Try LassoCV: which features does it switch off?
- Plot predicted against actual progression for the test set. Where are the biggest errors?